<a href="https://colab.research.google.com/github/e23254-cyber/Statistical-Learning-e23254/blob/main/data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import io
import scipy.stats as ss
from IPython.display import display, HTML

# Google Colab specific import
try:
    from google.colab import files
except ImportError:
    pass

# Plotly for interactive charts
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn for Normalization/Encoding
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [2]:
class PlottingMethods:
    """A modular class to handle granular chart generation using Plotly."""

    def __init__(self):
        pass

    def _wrap_html(self, fig):
        """Helper to wrap Plotly figures in HTML."""
        if fig is None:
            return {"status": "error", "html": "<p>Error generating plot.</p>"}
        return {"status": "success", "html": fig.to_html(full_html=False, include_plotlyjs='cdn')}

    def display_image(self, result):
        """Displays the HTML result from plotting methods."""
        if isinstance(result, dict) and "html" in result:
            display(HTML(result["html"]))
        else:
            print("Invalid plot result.")

    def plot_bar_chart(self, x, y, data, color=None, barmode='group', title=None):
        """Generates a bar chart."""
        try:
            fig = px.bar(data, x=x, y=y, color=color, barmode=barmode, title=title or f"Bar Chart: {y} by {x}")
            return self._wrap_html(fig)
        except Exception as e:
            print(f"Error plotting bar chart: {e}")
            return self._wrap_html(None)

    def plot_pie_chart(self, names, values, data, hole=0.0, title=None):
        """Generates a pie chart (or donut if hole > 0)."""
        try:
            fig = px.pie(data, names=names, values=values, hole=hole, title=title or f"Pie Chart: {values} by {names}")
            return self._wrap_html(fig)
        except Exception as e:
            print(f"Error plotting pie chart: {e}")
            return self._wrap_html(None)

    def plot_histogram(self, x, data, bins=None, title=None):
        """Generates a histogram."""
        try:
            fig = px.histogram(data, x=x, nbins=bins, title=title or f"Histogram of {x}")
            return self._wrap_html(fig)
        except Exception as e:
            print(f"Error plotting histogram: {e}")
            return self._wrap_html(None)

In [3]:
class DataInspector:
    """End-to-end tool for CSV data ingestion, cleaning, feature engineering, and visualization."""

    def __init__(self, df=None):
        self.df = df
        self.plotter = PlottingMethods()
        # Storage for transformed features
        self.numeric_transformed = None
        self.categorical_transformed = None

    # ==========================================
    # 1. Data Ingestion & Sanitization
    # ==========================================
    def upload_data(self):
        """Handles local file uploads in Google Colab and triggers auto-sanitization."""
        try:
            uploaded = files.upload()
            for fn in uploaded.keys():
                print(f"User uploaded file '{fn}' with length {len(uploaded[fn])} bytes")
                self.df = pd.read_csv(io.BytesIO(uploaded[fn]))
                self._sanitize_data()
                break # Just read the first uploaded file
        except Exception as e:
            print(f"Upload failed (Ensure you are in Google Colab): {e}")

    def _sanitize_data(self):
        """Replaces garbage strings with NaN and attempts automatic numeric conversion."""
        if self.df is None: return

        # Replace common garbage strings with actual NaNs
        garbage_strings = ['?', 'n/a', 'NULL', ' ', 'N/A', 'null', 'nan', 'NaN']
        self.df.replace(garbage_strings, np.nan, inplace=True)

        # Auto-type correction
        for col in self.df.columns:
            # Try to convert to numeric, if it fails, it leaves it as is
            converted = pd.to_numeric(self.df[col], errors='coerce')
            # Only update if the conversion doesn't result in an entirely null column (unless it was already null)
            if not converted.isna().all() or self.df[col].isna().all():
                self.df[col] = converted
        print("Data ingestion and sanitization complete.")

    # ==========================================
    # 2. Structural Analysis & Cleaning
    # ==========================================
    def get_summary(self):
        """Displays basic dataset structural summary."""
        if self.df is None: return print("No data loaded.")

        print(f"--- Dataset Structure ---")
        print(f"Rows: {self.df.shape[0]}, Columns: {self.df.shape[1]}")

        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")

        print("\n--- First 20 Rows ---")
        display(self.df.head(20))

    def handle_missing_values(self, strategy='median', fill_value=None):
        """Imputes missing values based on strategy: mean, median, mode, or constant."""
        if self.df is None: return

        for col in self.df.columns:
            if self.df[col].isnull().sum() == 0:
                continue

            if strategy == 'constant' and fill_value is not None:
                self.df[col].fillna(fill_value, inplace=True)
            elif strategy == 'mode':
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)
            elif pd.api.types.is_numeric_dtype(self.df[col]):
                if strategy == 'mean':
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
                elif strategy == 'median':
                    self.df[col].fillna(self.df[col].median(), inplace=True)

        print(f"Missing values handled using '{strategy}' strategy.")

    def remove_duplicates(self):
        """Prunes exact row matches."""
        if self.df is None: return
        initial_rows = len(self.df)
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_rows - len(self.df)} duplicate rows.")

    def handle_outliers(self, columns, find_and_delete=False):
        """IQR-based outlier detection and optional deletion for specific columns."""
        if self.df is None: return

        outlier_indices = set()
        for col in columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                outliers = self.df[(self.df[col] < lower_bound) | (self.df[col] > upper_bound)].index
                outlier_indices.update(outliers)

        if find_and_delete:
            self.df.drop(index=list(outlier_indices), inplace=True)
            print(f"Deleted {len(outlier_indices)} outlier rows based on IQR.")
        else:
            print(f"Found {len(outlier_indices)} outliers. Use find_and_delete=True to remove them.")

    def delete_rows(self):
        """Interactive method to delete rows based on index."""
        if self.df is None: return
        user_input = input("Enter comma-separated row indices to delete: ")
        try:
            indices = [int(i.strip()) for i in user_input.split(',')]
            self.df.drop(index=indices, inplace=True, errors='ignore')
            print(f"Deleted rows: {indices}")
        except ValueError:
            print("Invalid input. Please provide comma-separated numbers.")

    def delete_columns(self):
        """Interactive method to delete columns based on name."""
        if self.df is None: return
        user_input = input("Enter comma-separated column names to delete: ")
        cols = [c.strip() for c in user_input.split(',')]
        existing_cols = [c for c in cols if c in self.df.columns]
        self.df.drop(columns=existing_cols, inplace=True)
        print(f"Deleted columns: {existing_cols}")

    # ==========================================
    # 3. Feature Engineering Preparation
    # ==========================================
    def extract_normalized_numeric_data(self, method='standard'):
        """Scales numeric data using minmax, standard, or robust."""
        if self.df is None: return None
        num_cols = self.df.select_dtypes(include=[np.number]).columns
        if len(num_cols) == 0: return None

        scalers = {'minmax': MinMaxScaler(), 'standard': StandardScaler(), 'robust': RobustScaler()}
        if method not in scalers: method = 'standard'

        scaler = scalers[method]
        scaled_data = scaler.fit_transform(self.df[num_cols])
        self.numeric_transformed = pd.DataFrame(scaled_data, columns=num_cols, index=self.df.index)
        print(f"Numeric data scaled using {method}.")
        return self.numeric_transformed

    def extract_normalized_categorical_data(self, method='onehot'):
        """Encodes categorical data using onehot, ordinal, or uniform (scaled 0-1)."""
        if self.df is None: return None
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns
        if len(cat_cols) == 0: return None

        if method == 'onehot':
            encoder = OneHotEncoder(sparse_output=False, drop='first')
            encoded_data = encoder.fit_transform(self.df[cat_cols])
            cols = encoder.get_feature_names_out(cat_cols)
            self.categorical_transformed = pd.DataFrame(encoded_data, columns=cols, index=self.df.index)
        elif method == 'ordinal' or method == 'uniform':
            encoder = OrdinalEncoder()
            encoded_data = encoder.fit_transform(self.df[cat_cols])
            df_enc = pd.DataFrame(encoded_data, columns=cat_cols, index=self.df.index)
            if method == 'uniform':
                scaler = MinMaxScaler()
                df_enc[cat_cols] = scaler.fit_transform(df_enc)
            self.categorical_transformed = df_enc

        print(f"Categorical data encoded using {method}.")
        return self.categorical_transformed

    def create_normalized_data_df(self):
        """Creates a unified DataFrame containing all transformed data."""
        if self.df is None: return None

        parts = []
        if self.numeric_transformed is not None:
            parts.append(self.numeric_transformed)
        if self.categorical_transformed is not None:
            parts.append(self.categorical_transformed)

        if parts:
            unified_df = pd.concat(parts, axis=1)
            print("Unified Normalized DataFrame created.")
            return unified_df
        return self.df

    # ==========================================
    # 4. Advanced Interactive Visualization
    # ==========================================
    def plot_numerical(self, columns):
        """Generates a 3-panel subplot: Violin/Box, Scatter, and Histogram for numeric columns."""
        if self.df is None: return

        for col in columns:
            if col not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[col]):
                continue

            fig = make_subplots(rows=1, cols=3, subplot_titles=(f"Violin: {col}", f"Scatter: Index vs {col}", f"Histogram: {col}"))

            # 1. Violin Plot
            fig.add_trace(go.Violin(x=self.df[col], name=col, orientation='h'), row=1, col=1)
            # 2. Scatter Plot
            fig.add_trace(go.Scatter(x=self.df.index, y=self.df[col], mode='markers', name=col), row=1, col=2)
            # 3. Histogram
            fig.add_trace(go.Histogram(x=self.df[col], name=col), row=1, col=3)

            fig.update_layout(title_text=f"Univariate Analysis: {col}", showlegend=False, height=400)
            fig.show()

    def plot_relationship(self, col1, col2):
        """Smart relationship plotter based on data types."""
        if self.df is None or col1 not in self.df.columns or col2 not in self.df.columns: return

        is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
        is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])

        if is_num1 and is_num2:
            # Num-Num: Scatter with OLS Trendline
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Scatter: {col1} vs {col2}")
        elif not is_num1 and not is_num2:
            # Cat-Cat: Grouped Bar chart
            df_counts = self.df.groupby([col1, col2]).size().reset_index(name='Count')
            fig = px.bar(df_counts, x=col1, y='Count', color=col2, barmode='group', title=f"Grouped Bar: {col1} vs {col2}")
        else:
            # Cat-Num: Box plot with all data points
            c_cat, c_num = (col1, col2) if not is_num1 else (col2, col1)
            fig = px.box(self.df, x=c_cat, y=c_num, points="all", title=f"Box Plot: {c_num} distribution by {c_cat}")

        fig.show()

    def plot_categorical(self, columns):
        """Bar charts displaying both raw counts and percentage labels."""
        if self.df is None: return

        for col in columns:
            if col in self.df.columns:
                counts = self.df[col].value_counts().reset_index()
                counts.columns = [col, 'Count']
                counts['Percentage'] = (counts['Count'] / counts['Count'].sum() * 100).round(2).astype(str) + '%'

                fig = px.bar(counts, x=col, y='Count', text='Percentage', title=f"Categorical Frequency: {col}")
                fig.update_traces(textposition='outside')
                fig.show()

    # ==========================================
    # 5. Deep Statistical Insights
    # ==========================================
    def plot_all_associations_heatmap(self):
        """Visualizes relationships across all data types using Pearson, Cramér's V, and Correlation Ratio."""
        if self.df is None: return

        cols = self.df.columns
        n = len(cols)
        assoc_matrix = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)

        for i in range(n):
            for j in range(i+1, n):
                col1, col2 = cols[i], cols[j]
                is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
                is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])

                score = 0
                # Filter out NaNs for pairwise calculation
                valid_mask = self.df[col1].notna() & self.df[col2].notna()
                v1, v2 = self.df.loc[valid_mask, col1], self.df.loc[valid_mask, col2]

                if len(v1) == 0:
                    score = np.nan
                elif is_num1 and is_num2:
                    # Pearson's r
                    score, _ = ss.pearsonr(v1, v2)
                elif not is_num1 and not is_num2:
                    # Cramér's V
                    confusion_matrix = pd.crosstab(v1, v2)
                    chi2 = ss.chi2_contingency(confusion_matrix)[0]
                    total = confusion_matrix.sum().sum()
                    phi2 = chi2 / total
                    r, k = confusion_matrix.shape
                    if r > 1 and k > 1:
                        phi2corr = max(0, phi2 - ((k-1)*(r-1))/(total-1))
                        rcorr = r - ((r-1)**2)/(total-1)
                        kcorr = k - ((k-1)**2)/(total-1)
                        score = np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))
                    else:
                        score = 0
                else:
                    # Mixed: Correlation Ratio (Eta)
                    c_num, c_cat = (v1, v2) if is_num1 else (v2, v1)
                    cats = c_cat.unique()
                    y_avg = c_num.mean()
                    numerator = sum([len(c_num[c_cat == cat]) * (c_num[c_cat == cat].mean() - y_avg)**2 for cat in cats])
                    denominator = sum((c_num - y_avg)**2)
                    score = np.sqrt(numerator / denominator) if denominator != 0 else 0

                assoc_matrix.iloc[i, j] = score
                assoc_matrix.iloc[j, i] = score

        fig = px.imshow(assoc_matrix, text_auto=".2f", color_continuous_scale='RdBu_r',
                        zmin=-1, zmax=1, title="Unified Association Heatmap (Pearson / Cramér's V / Eta)")
        fig.show()

In [4]:
# 1. Initialize
inspector = DataInspector()

# 2. Upload Data (In Colab, this will prompt a file upload dialog)
# inspector.upload_data()

# -- For Quick Testing, Let's inject the Titanic Dataset --
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(url)
inspector._sanitize_data() # Apply custom NaN handlers

# 3. Structural Analysis & Cleaning
inspector.get_summary()
inspector.handle_missing_values(strategy='median')
inspector.remove_duplicates()

# 4. Feature Engineering (Normalization)
inspector.extract_normalized_numeric_data(method='robust')
inspector.extract_normalized_categorical_data(method='onehot')
final_df = inspector.create_normalized_data_df()

# 5. Advanced Visualizations
inspector.plot_relationship('Age', 'Fare')       # Num-Num (Scatter + OLS)
inspector.plot_relationship('Survived', 'Age')   # Cat-Num (Box Plot)
inspector.plot_all_associations_heatmap()        # Unified Heatmap

Data ingestion and sanitization complete.
--- Dataset Structure ---
Rows: 891, Columns: 12
Numerical Columns (8): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
Categorical Columns (4): ['Name', 'Sex', 'Cabin', 'Embarked']

--- First 20 Rows ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


Missing values handled using 'median' strategy.
Removed 0 duplicate rows.
Numeric data scaled using robust.
Categorical data encoded using onehot.
Unified Normalized DataFrame created.


/tmp/ipykernel_3317/1725145281.py:77: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df[col].fillna(self.df[col].median(), inplace=True)


/tmp/ipykernel_3317/1725145281.py:283: RuntimeWarning:

divide by zero encountered in scalar divide

/tmp/ipykernel_3317/1725145281.py:283: RuntimeWarning:

invalid value encountered in scalar divide

/tmp/ipykernel_3317/1725145281.py:283: RuntimeWarning:

invalid value encountered in scalar divide

